# 01. 룰북 PDF -> 벡터 DB 만들기 (RAG)

**목적**: 마피아 게임 AI 플레이어가 참고할 지식을 PDF 한 권으로 만들고, 검색 가능한 벡터 DB(Chroma)로 구축한다.

PDF 16쪽의 구성은 다음과 같다.

| 쪽 | 내용 | type | role | name |
|---|---|---|---|---|
| 1~4 | 룰북 (규칙, 직업 능력, 진행 순서) | rule | all | - |
| 5~8 | 직업별 전략 가이드 | strategy | mafia / police / doctor / citizen | - |
| 9~16 | 캐릭터 프로필 8명 | character | all | 캐릭터 이름 |

**핵심 아이디어: 페이지마다 메타데이터(type, role, name)를 붙인다.**
그러면 "마피아 AI는 마피아 전략 페이지만", "강태호 AI는 강태호 프로필만" 처럼 검색 범위를 **필터로 제한**할 수 있다.
유사도 검색만 쓰면 시민 AI가 마피아 전략 문서를 읽어버리는 정보 유출이 생길 수 있다.

진행 순서: PDF 로딩 -> 메타데이터 확인 -> 청킹 -> 벡터 DB 구축 -> 필터 검색 시연

In [1]:
# ===== 0. 작업 폴더 맞추기 =====
# 이 노트북은 08_Mafia_Agent 폴더에서 실행한다고 가정한다.
# (VS Code에서 상위 폴더를 열었을 때도 동작하도록 폴더를 찾아 이동한다.)
import os
import sys
from pathlib import Path

if not Path("config.py").exists():
    if Path("08_Mafia_Agent/config.py").exists():
        os.chdir("08_Mafia_Agent")
    else:
        raise FileNotFoundError("08_Mafia_Agent 폴더에서 노트북을 열어주세요.")

sys.path.insert(0, os.getcwd())

import config

print("===== 작업 폴더 =====")
print(config.BASE_DIR)
print("LLM:", config.LLM_MODEL, "/ 임베딩:", config.EMBED_MODEL)

===== 작업 폴더 =====
.
LLM: qwen3:8b / 임베딩: bge-m3


## 1. PDF 로딩

`PyPDFLoader`는 PDF를 **페이지 한 장 = Document 하나**로 읽는다. 각 Document의 `metadata["page"]`는 0부터 시작한다.

In [2]:
# ===== 1. PDF 로딩 =====
from langchain_community.document_loaders import PyPDFLoader

raw_docs = PyPDFLoader(str(config.PDF_PATH)).load()

print("===== PDF 로딩 완료 =====")
print("PDF 경로:", config.PDF_PATH)
print("PDF 페이지 수:", len(raw_docs))
print("1쪽 일부:")
print(raw_docs[0].page_content[:150])
print("원래 metadata:", raw_docs[0].metadata)

===== PDF 로딩 완료 =====
PDF 경로: .\data\마피아_룰북.pdf
PDF 페이지 수: 16
1쪽 일부:
마피아 게임 룰북
마피아는 숨겨진 역할을 가진 8명이 토론과 투표로 서로의 정체를 추리하는 게임입니다. 시민팀
은 마피아를 모두 찾아내야 하고, 마피아팀은 정체를 숨기며 시민을 줄여 나갑니다.
게임 개요
인원: 8명 (마피아 2, 경찰 1, 의사 1, 시민 4)
역할은
원래 metadata: {'producer': 'Skia/PDF m156', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/156.0.0.0 Safari/537.36', 'creationdate': '2026-11-17T02:09:53+00:00', 'title': '마피아 룰북', 'moddate': '2026-11-17T02:09:53+00:00', 'source': 'C:\\Users\\USER\\Desktop\\수업\\딥러닝\\My_Langchain\\08_Mafia_Agent\\data\\마피아_룰북.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1'}


## 2. 페이지 메타데이터 부착

`rag.load_pdf_with_meta()`는 `config.PAGE_META`의 type / role / name / title을 각 페이지에 붙인다.
이때 **PDF 각 페이지의 첫 줄이 약속된 제목과 같은지 검증**해서, PDF가 바뀌어 메타데이터가 어긋나는 사고를 막는다.

In [3]:
# ===== 2. 메타데이터 부착 + 표로 확인 =====
import rag

docs = rag.load_pdf_with_meta()

print("%-4s %-10s %-8s %-8s %s" % ("쪽", "type", "role", "name", "제목"))
print("-" * 60)
for d in docs:
    m = d.metadata
    print("%-4d %-10s %-8s %-8s %s" % (m["page"] + 1, m["type"], m["role"], m["name"], m["title"]))

쪽    type       role     name     제목
------------------------------------------------------------
1    rule       all      -        마피아 게임 룰북
2    rule       all      -        기본 규칙과 승리 조건
3    rule       all      -        직업별 능력
4    rule       all      -        진행 순서와 특수 규칙
5    strategy   mafia    -        마피아 전략 가이드
6    strategy   police   -        경찰 전략 가이드
7    strategy   doctor   -        의사 전략 가이드
8    strategy   citizen  -        시민 전략 가이드
9    character  all      강태호      캐릭터 프로필: 강태호
10   character  all      윤서진      캐릭터 프로필: 윤서진
11   character  all      박준수      캐릭터 프로필: 박준수
12   character  all      최민지      캐릭터 프로필: 최민지
13   character  all      정도현      캐릭터 프로필: 정도현
14   character  all      한소율      캐릭터 프로필: 한소율
15   character  all      오지환      캐릭터 프로필: 오지환
16   character  all      임채원      캐릭터 프로필: 임채원


## 3. 청킹 (Chunking)

페이지가 길면 한 덩어리로 검색하기 부정확하므로 `RecursiveCharacterTextSplitter`로 **500자 단위(겹침 50자)**로 자른다.
쪼개진 chunk도 원래 페이지의 메타데이터를 그대로 물려받는다. (필터가 chunk에서도 동작해야 하므로 중요하다.)

In [4]:
# ===== 3. 청킹 =====
from collections import Counter

chunks = rag.split_docs(docs, chunk_size=500, chunk_overlap=50)

print("===== 청킹 결과 =====")
print("페이지 수:", len(docs), "-> chunk 수:", len(chunks))

# type 별 chunk 수
by_type = Counter(c.metadata["type"] for c in chunks)
for t, n in by_type.items():
    print("  %-10s %d개" % (t, n))

# 쪽별 chunk 수 표
per_page = Counter(c.metadata["page"] + 1 for c in chunks)
print("\n쪽별 chunk 수:", dict(sorted(per_page.items())))

# 미리보기: 마피아 전략 가이드의 첫 chunk
preview = [c for c in chunks if c.metadata["type"] == "strategy" and c.metadata["role"] == "mafia"][0]
print("\n[미리보기] metadata:", preview.metadata)
print(preview.page_content[:200])

===== 청킹 결과 =====
페이지 수: 16 -> chunk 수: 23
  rule       6개
  strategy   6개
  character  11개

쪽별 chunk 수: {1: 2, 2: 1, 3: 1, 4: 2, 5: 2, 6: 2, 7: 1, 8: 1, 9: 1, 10: 2, 11: 2, 12: 1, 13: 1, 14: 1, 15: 1, 16: 2}

[미리보기] metadata: {'producer': 'Skia/PDF m156', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/156.0.0.0 Safari/537.36', 'creationdate': '2026-11-17T02:09:53+00:00', 'title': '마피아 전략 가이드', 'moddate': '2026-11-17T02:09:53+00:00', 'source': 'C:\\Users\\USER\\Desktop\\수업\\딥러닝\\My_Langchain\\08_Mafia_Agent\\data\\마피아_룰북.pdf', 'total_pages': 16, 'page': 4, 'page_label': '5', 'type': 'strategy', 'role': 'mafia', 'name': '-', 'chunk_idx': 0}
마피아 전략 가이드
마피아 팀(2명)이 이기기 위한 행동 요령입니다. 이 내용은 마피아만 참고합니다.
행동 요령
시민처럼 말하세요. 증거 없는 확신, 지나치게 조용함, 지나치게 흥분하는 것은 의심을 삽니
다.
동료 마피아를 너무 티 나게 감싸지 마세요. 가끔은 동료를 가볍게 의심하는 발언도 섞어서
거리를 두세요.
투표에서 동료가 몰리면 같이 몰아가지 말고,


## 4. 벡터 DB 구축 (Chroma + bge-m3 임베딩)

chunk 텍스트를 `bge-m3`(Ollama) 임베딩으로 벡터로 바꿔 Chroma에 저장한다. **Ollama 앱이 켜져 있고 `bge-m3`가 설치되어 있어야 한다.**
`rag.build_db()`는 기존 DB를 지우고 새로 만든다 (`reset=True`).

In [5]:
# ===== 4. 벡터 DB 구축 =====
vs = rag.build_db()

print("===== 벡터 DB 구축 완료 =====")
print("저장 위치:", config.DB_DIR)
print("저장된 Chunk 수:", vs._collection.count())

===== 벡터 DB 구축 완료 =====
저장 위치: .\vector_db
저장된 Chunk 수: 23


## 5. 메타데이터 필터 검색 시연

같은 DB에서 **필터 조건만 바꿔** 3가지 용도로 검색한다.

| 용도 | 함수 | 필터 |
|---|---|---|
| 사회자가 규칙 질문에 답할 때 | `search_rules` | type = rule |
| AI가 자기 역할의 전략 조언을 볼 때 | `search_strategy` | type = strategy AND role = 내 역할 |
| AI가 자기 캐릭터 설정을 읽을 때 | `get_persona` | type = character AND name = 내 이름 |

In [6]:
# ===== 5-1. 규칙 검색 (type = rule) =====
question = "의사가 자기 자신을 살릴 수 있나요?"

print("질문:", question)
for i, d in enumerate(rag.search_rules(vs, question, k=3), 1):
    print("\n[%d] 쪽 %d | %s | type=%s" % (i, d.metadata["page"] + 1, d.metadata["title"], d.metadata["type"]))
    print(d.page_content[:150].replace("\n", " "))

질문: 의사가 자기 자신을 살릴 수 있나요?

[1] 쪽 3 | 직업별 능력 | type=rule
직업별 능력 마피아 (2명) 매일 밤 시민팀 한 명을 살해 대상으로 정합니다. 두 마피아가 순서대로 결정합니다. 첫 마피아 가 대상과 이유를 제안하면, 두 번째 마피아는 동료의 제안을 보고 동의하거나 바꿔서 최종 결정 을 합니다. 마피아가 한 명만 살아있으면 그 사람의

[2] 쪽 4 | 진행 순서와 특수 규칙 | type=rule
진행 순서와 특수 규칙 하루의 진행 밤: 마피아가 살해 대상 결정 → 의사가 보호 → 경찰이 조사. 대상이 같으면 아무도 죽지 않습 니다. 아침: 사망자와 그 직업을 발표합니다. 승리 조건을 확인합니다. 낮 토론: 살아있는 사람이 무작위 순서로 한 명씩 1인 1회 발언

[3] 쪽 2 | 기본 규칙과 승리 조건 | type=rule
기본 규칙과 승리 조건 역할 구성 총 8명: 마피아 2명, 경찰 1명, 의사 1명, 시민 4명. 마피아 2명은 서로가 누구인지 압니다. 다른 역할은 자기 정체만 압니다. 경찰의 조사 결과는 경찰 본인만 압니다. 승리 조건 시민팀 승리: 살아있는 마피아가 0명이 되면 시


In [7]:
# ===== 5-2. 전략 검색: 같은 질문, 역할만 다르게 (type = strategy AND role) =====
question = "초반 토론에서 어떻게 행동해야 하나요?"

for role in ["mafia", "citizen"]:
    print("\n##### role=%s (%s) #####" % (role, config.ROLE_KO[role]))
    for d in rag.search_strategy(vs, role, question, k=2):
        print("- 쪽 %d | %s | role=%s" % (d.metadata["page"] + 1, d.metadata["title"], d.metadata["role"]))
        print("  ", d.page_content[:100].replace("\n", " "))


##### role=mafia (마피아) #####
- 쪽 5 | 마피아 전략 가이드 | role=mafia
   마피아 전략 가이드 마피아 팀(2명)이 이기기 위한 행동 요령입니다. 이 내용은 마피아만 참고합니다. 행동 요령 시민처럼 말하세요. 증거 없는 확신, 지나치게 조용함, 지나치게 흥
- 쪽 5 | 마피아 전략 가이드 | role=mafia
   많은 시민에게 의심을 돌리세요. 투표 때는 다수가 향하는 쪽에 합류해 눈에 띄지 않도록 하세요.

##### role=citizen (시민) #####
- 쪽 8 | 시민 전략 가이드 | role=citizen
   시민 전략 가이드 시민은 밤 능력이 없으므로 토론과 투표가 무기입니다. 이 내용은 시민만 참고합니다. 행동 요령 모든 발언을 기억하고, 앞뒤가 맞지 않는 말(모순)을 찾으세요. 투


In [8]:
# ===== 5-3. 필터가 없으면 어떻게 될까? (비교) =====
question = "초반 토론에서 어떻게 행동해야 하나요?"

print("[필터 없음] 전체 문서에서 유사도 검색")
for d in vs.similarity_search(question, k=4):
    print("- 쪽 %d | %s | type=%s role=%s" % (d.metadata["page"] + 1, d.metadata["title"], d.metadata["type"], d.metadata["role"]))
print("\n-> 필터가 없으면 다른 역할의 전략/캐릭터 문서가 섞여 들어올 수 있다.")

[필터 없음] 전체 문서에서 유사도 검색
- 쪽 5 | 마피아 전략 가이드 | type=strategy role=mafia
- 쪽 5 | 마피아 전략 가이드 | type=strategy role=mafia
- 쪽 4 | 진행 순서와 특수 규칙 | type=rule role=all
- 쪽 11 | 캐릭터 프로필: 박준수 | type=character role=all

-> 필터가 없으면 다른 역할의 전략/캐릭터 문서가 섞여 들어올 수 있다.


In [9]:
# ===== 5-4. 캐릭터 페르소나 (type = character AND name) =====
for name in ["강태호", "윤서진"]:
    print("\n##### %s #####" % name)
    print(rag.get_persona(vs, name)[:250])


##### 강태호 #####
캐릭터 프로필: 강태호
공격형
한 줄 소개
돌직구로 마피아를 압박하는 행동파
성격
솔직하고 승부욕이 강합니다. 의심이 들면 바로 입 밖에 내고, 돌려 말하는 것을 싫어합니다.
말투 규칙
반말을 씁니다. 문장은 짧고 단호하며 "솔직히 말해서", "야," "뻔하잖아" 같은 말을 자주 씁니다.
존댓말은 쓰지 않습니다.
예시 대사
평소: "오늘은 일단 얘기부터 들어보자. 다들 말 아끼지 마."
의심할 때: "야, 너 아까 말이랑 지금 말이 다르잖아. 

##### 윤서진 #####
캐릭터 프로필: 윤서진
논리형
한 줄 소개
근거를 차분히 정리하는 분석가
성격
침착하고 신중합니다. 감정보다 사실과 순서를 중시하며, 발언을 번호나 근거로 정리하는 것을
좋아합니다.
말투 규칙
차분한 존댓말(~습니다, ~입니다)을 씁니다. 문장은 중간 길이이며 "근거를 정리하면", "논리적
으로 보면", "첫째, 둘째" 같은 표현을 씁니다.
예시 대사
평소: "현재까지의 발언을 정리해 보겠습니다. 아직 결정적인 근거는 없습니다."
의심할 때: 


In [10]:
print("===== 01 완료: 이제 02_agent_test.ipynb 로 이동하세요 =====")

===== 01 완료: 이제 02_agent_test.ipynb 로 이동하세요 =====
